### Root the tree and write internal nodes. Model of the tree: 
- Akaike Information Criterion: LG+F+R10
- Corrected Akaike Information Criterion: LG+G4
- Bayesian Information Criterion: LG+F+R10
- Best-fit model: LG+F+R10 chosen according to BIC

In [4]:
from ete4 import PhyloTree

t = PhyloTree(open("PF13853_9606.7764.mafft.lg.treefile"), parser=1)

A = 'New|7764.A0A8C4N3S4'
t.set_outgroup(A)

t.name = "node_0"
i = 1
for n in t.traverse():
    if n is t or n.is_leaf:          # root already named
        continue
    n.name = f"node_{i}"
    i +=1


t.write(parser=1, outfile="PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames", format_root_node=True)

t.explore()

Existing explorer available at http://127.0.0.1:5000


In [5]:
# print all the nodes written

nodes = []

for n in t.traverse():
    if not n.is_leaf:
        nodes.append(n.name)

In [6]:
sorted_nodes = sorted(nodes, key=lambda x: int(x.split("_")[1]))

In [8]:
len(sorted_nodes)

433

### ASR

In [9]:
import os
from pathlib import Path, PurePath
import subprocess, shutil
from Bio import SeqIO
import re

In [10]:
msa_path = Path("PF13853_9606.7764.mafft")
tree_path = Path("PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames")
presence_absence_path = Path(str(msa_path) + ".presence_absence")
state_file = f"{presence_absence_path}.state" 

In [11]:
msa = open(msa_path, "r").read().split(">")[1:]
with open(presence_absence_path, "w") as out:
    for i in msa:
        out.write(">" + i.split("\n")[0] + "\n")
        sequence = i.split("\n", 1)[1].replace("\n", "")
        for s in sequence:
            out.write("0" if s == "-" else "1")
        out.write("\n")

#presence-absence file        
subprocess.run(["iqtree", "-m", "MFP", "-s", presence_absence_path,
                "-te", tree_path, "-asr", "-nt", "12"], check=True)

IQ-TREE version 3.0.1 for Linux x86 64-bit built Jul  9 2025
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    nada-P8 (AVX512, FMA3, 125 GB RAM)
Command: iqtree -m MFP -s PF13853_9606.7764.mafft.presence_absence -te PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames -asr -nt 12
Seed:    44877 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Jul 29 17:26:22 2025
Kernel:  AVX+FMA - 12 threads (64 CPU cores detected)

Reading alignment file PF13853_9606.7764.mafft.presence_absence ... Fasta format detected
Reading fasta file: done in 0.00125871 secs using 72.14% CPU
Alignment most likely contains binary sequences
Constructing alignment: done in 0.000781652 secs using 1325% CPU
Alignment has 434 sequences with 337 columns, 108 distinct patterns
208 parsimony-informative, 69 singleton sites, 60 constant 

CompletedProcess(args=['iqtree', '-m', 'MFP', '-s', PosixPath('PF13853_9606.7764.mafft.presence_absence'), '-te', PosixPath('PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames'), '-asr', '-nt', '12'], returncode=0)

In [12]:
def run_asr_for_node(node_name, original_msa, tree_file):
    node_folder = f"NODE_{node_name}"
    Path(node_folder).mkdir(exist_ok=True)

    # 1. 0/1 mask
    os.system(f"python asr_extract.py {presence_absence_path}.state {node_name}")
    shutil.move(f"{node_name}_ASR.fasta", f"{node_folder}/{node_name}_mask.fasta")
    asr_seq = open(f"{node_folder}/{node_name}_mask.fasta").read().split('\n',1)[1].strip()

    # 2. filtered alignment
    filt = f"{node_folder}/human_sequences.{node_name}.fasta"
    with open(filt, "w") as out:
        for rec in SeqIO.parse(original_msa, "fasta"):
            seq = str(rec.seq)
            if '1' not in asr_seq:                                 # root fix
                kept = seq
            else:
                kept = ''.join(s for s, a in zip(seq, asr_seq) if a == '1')
            out.write(f">{rec.id}\n{kept}\n")

    # 3. amino-acid ASR
    os.system(
        f"iqtree -s {filt} -m LG+F+R9 -te {tree_file} "
        f"-asr -nt 48"
    )

    # 4. extract ancestral protein
    os.system(f"python asr_extract.py {filt}.state {node_name}")
    shutil.move(f"{node_name}_ASR.fasta", f"{node_folder}/{node_name}_ASR.final.fasta")

In [13]:
with open(tree_path) as tr:
    nodes = sorted(set(re.findall(r'node_\d+', tr.read())))

for node in nodes:
    run_asr_for_node(node_name=node,
                     original_msa=msa_path,
                     tree_file=tree_path)

with open("all_nodes_ASR.final.fasta", "w") as out:
    for f in sorted(Path(".").glob("NODE_*/node_*_ASR.final.fasta")):
        out.write(f.read_text())


>node_0_ASR


Protein length: 0
Mean posterior probability = nan (standard deviation = nan, median = nan)

IQ-TREE version 3.0.1 for Linux x86 64-bit built Jul  9 2025
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    nada-P8 (AVX512, FMA3, 125 GB RAM)
Command: iqtree -s NODE_node_0/human_sequences.node_0.fasta -m LG+F+R9 -te PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames -asr -nt 48
Seed:    713068 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Jul 29 17:31:21 2025
Kernel:  AVX+FMA - 48 threads (64 CPU cores detected)

Reading alignment file NODE_node_0/human_sequences.node_0.fasta ... Fasta format detected
Reading fasta file: done in 0.00127424 secs using 88.37% CPU
Alignment most likely contains protein sequences
Constructing alignment: done in 0.00164625 secs using 5486% CPU
Alignment has 


>node_1_ASR
0000000111110011111111111111111011111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110111111111111111111111111111111111111111111111111111111111011111111111111111111111111111111111111100011111111111111111111111111111111111111111111100000000000

Protein length: 337
Mean posterior probability = 0.996 (standard deviation = 0.033, median = 1.0)

IQ-TREE version 3.0.1 for Linux x86 64-bit built Jul  9 2025
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    nada-P8 (AVX512, FMA3, 125 GB RAM)
Command: iqtree -s NODE_node_1/human_sequences.node_1.fasta -m LG+F+R9 -te PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames -asr -nt 48
Seed:    663564 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Jul 29 17:3